In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv("ml_dataset.csv")

print(df.shape)
print(df.head())

(8255, 67)
  delivery_day  hour  price_value  buy_volume_value  sell_volume_value  \
0   2025-10-01     0      66.0275           17033.2            25649.9   
1   2025-10-01     1      64.3350           16741.5            24200.2   
2   2025-10-01     2      63.5200           17129.7            23233.6   
3   2025-10-01     3      63.7525           16975.3            23914.6   
4   2025-10-01     4      63.8350           17059.1            24112.0   

   net_volume  flow_DK1_export  flow_DK1_import  flow_GER_export  \
0     -8616.7           1241.6              0.0           1444.0   
1     -7458.7            821.1              0.0           1444.0   
2     -6103.9           1085.8              0.0           1444.0   
3     -6939.3           1305.6              0.0           1444.0   
4     -7052.9           1324.3              0.0           1444.0   

   flow_GER_import  ...  price_BE  price_FR  Solar  Wind Onshore  \
0              0.0  ...   88.7325   79.5300    0.0      892.9225   

In [3]:
print(df.columns)

Index(['delivery_day', 'hour', 'price_value', 'buy_volume_value',
       'sell_volume_value', 'net_volume', 'flow_DK1_export', 'flow_DK1_import',
       'flow_GER_export', 'flow_GER_import', 'flow_NL_export',
       'flow_NL_import', 'flow_NO1_export', 'flow_NO1_import',
       'flow_NO5_export', 'flow_NO5_import', 'capacity_GER', 'capacity_NL',
       'capacity_NO1', 'capacity_NO5', 'util_GER', 'util_NL', 'util_NO1',
       'util_NO5', 'capacity_missing_DK1', 'capacity_missing_GER',
       'capacity_missing_NL', 'capacity_missing_NO1', 'capacity_missing_NO5',
       'capacity_zero_DK1', 'capacity_zero_GER', 'capacity_zero_NL',
       'capacity_zero_NO1', 'capacity_zero_NO5', 'price_DK1', 'price_NO1',
       'price_NO5', 'total_exports', 'total_imports', 'net_flow',
       'day_of_week', 'month', 'day_of_year', 'day_sin', 'day_cos', 'year_sin',
       'year_cos', 'price_lag_24', 'price_lag_168', 'price_DK1_lag24',
       'price_NO1_lag24', 'price_NO5_lag24', 'price_DK1_lag168',
       

In [4]:
df["delivery_day"] = pd.to_datetime(df["delivery_day"])

#optionally filter out data before 2025-01-01
df = df[df["delivery_day"] >= "2020-01-01"]

# Target: price 24 hours ahead
df = df.sort_values(["delivery_day", "hour"]).reset_index(drop=True)

df["target_price_next_day"] = df["price_value"].shift(-24)

# Drop rows without a future price (last 24 rows)
df = df.dropna(subset=["target_price_next_day"])

print(df.shape)
df[["delivery_day", "hour", "price_value", "target_price_next_day"]].head(10)

(8231, 68)


,delivery_day,hour,price_value,target_price_next_day
0,2025-10-01,0,66.0275,66.3200
1,2025-10-01,1,64.3350,66.2350
2,2025-10-01,2,63.5200,64.0425
3,2025-10-01,3,63.7525,63.4150
4,2025-10-01,4,63.8350,64.2650
5,2025-10-01,5,66.2350,66.8275
6,2025-10-01,6,69.4675,69.9750
7,2025-10-01,7,81.3125,77.6975
8,2025-10-01,8,83.6575,82.0700
9,2025-10-01,9,75.0250,73.7875


In [14]:
drop_cols = ['delivery_day', 'net_volume', 'total_exports', 'flow_NO1_import', 'day_sin', 'Forecasted Generation', 'flow_NL_export', 'buy_volume_value', 'hour', 'total_imports', 'price_NO1_lag24', 'flow_DK1_import', 'flow_NO5_import', 'flow_GER_import', 'sell_volume_value', 'flow_NL_import', 'capacity_zero_NO1', 'capacity_zero_NO5', 'capacity_NO1', 'capacity_GER', 'capacity_NL', 'capacity_zero_NL', 'capacity_zero_GER', 'capacity_missing_NO5', 'capacity_zero_DK1', 'capacity_missing_NO1', 'capacity_missing_NL', 'util_NO1', 'capacity_missing_GER', 'capacity_missing_DK1', 'util_NO5', 'util_GER', 'util_NL', 'capacity_NO5', 'Solar']

X = df.drop(columns=drop_cols + ["target_price_next_day"])
y = df["target_price_next_day"]

print("Features:", len(X.columns))

split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print(f"Train rows: {len(X_train):,}")
print(f"Test rows : {len(X_test):,}")

Features: 32
Train rows: 6,584
Test rows : 1,647


In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

print("Training...")
model.fit(X_train, y_train)

print("Predicting...")
pred = model.predict(X_test)
print("done")

In [ ]:
rmse = root_mean_squared_error(y_test, pred)

print(f"RMSE = {rmse:.4f}")

r2 = r2_score(
    y_test,
    pred
)

print("R²:", r2)

rmse = root_mean_squared_error(
y_test,
X_test["price_lag24"]
)
print(f"price_lag24 RMSE: {rmse:.4f}")

In [15]:
model = XGBRegressor(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.01,
        objective="reg:squarederror"
    )

print("Training...")
model.fit(X_train, y_train)

print("Predicting...")
pred = model.predict(X_test)
print("done")

Training...
Predicting...
done


In [16]:
rmse = root_mean_squared_error(y_test, pred)

print(f"RMSE = {rmse:.4f}")

r2 = r2_score(
    y_test,
    pred
)

print("R²:", r2)

rmse = root_mean_squared_error(
y_test,
X_test["price_lag_24"]
)
print(f"price_lag_24 RMSE: {rmse:.4f}")

RMSE = 26.1474
R²: 0.5738047939334462
price_lag_24 RMSE: 39.1043


In [17]:
importance = (
pd.DataFrame({
"feature": X.columns,
"importance": model.feature_importances_
})
.sort_values("importance", ascending=False)
)
print(importance[importance["importance"] < 0.01]["feature"].tolist())
print(importance[importance["importance"] > 0.01])

[]
                    feature  importance
0               price_value    0.214471
6                 price_NO1    0.083089
22                 price_NL    0.056141
9               day_of_week    0.047177
11              day_of_year    0.046121
31  no5_reservoir_level_MWh    0.039574
23              price_DE_LU    0.039254
10                    month    0.035733
24                 price_BE    0.033171
13                 year_sin    0.029496
30  no1_reservoir_level_MWh    0.029142
29  no2_reservoir_level_MWh    0.028610
25                 price_FR    0.027511
14                 year_cos    0.027081
28          Forecasted Load    0.021815
21         price_NO5_lag168    0.019858
15             price_lag_24    0.018858
19         price_DK1_lag168    0.017540
3           flow_NO1_export    0.017206
7                 price_NO5    0.015087
12                  day_cos    0.014810
4           flow_NO5_export    0.014461
18          price_NO5_lag24    0.013625
1           flow_DK1_export    0.0134